# POIデータから地域特徴量を作る：街を数値で表してみる

地域特徴量は、街を比較するための「数値プロフィール」です。**1地域＝1行、特徴量＝列**の表を作ります。今回は「吉祥寺駅周辺の実験用矩形」1行だけ。次段階では同じ列を複数地域で計算し、類似度を求めます。

250m・500m格子は算出根拠となる途中データです。まず地域全体の表を見て、その後に格子から密度・集中度を求めた方法を確認します。

[列の日本語名・単位・定義](../docs/region-feature-columns.md) / [方法](../docs/region-features.md) / [検証](../docs/region-features-validation.md) / [記事資料](../docs/region-features-article-handoff.md)

## 1. 準備と固定条件

`projects/poi/` またはその `notebooks/` から実行します。既存の `osm`・`notebooks` 依存グループを使います。

- 入力はconfigs/region_features.tomlの実行IDとSHA-256で固定
- 第IX系 EPSG:6677、東・北の座標、共通原点(0,0)
- 格子の内部境界は東側・北側に一意割当、AOI外周は包含
- 密度の分母はAOIと交差する実面積
- 構成比の分母はカテゴリ欠損も含む全登録件数。営業状態による除外なし
- taxonomyの別名・階層は重複計上せず、単一basic_categoryを使用
- 0件区画の構成比は未定義。未取得データは0件とせず停止

In [ ]:
from geoai_open_lab.region_features import load_inputs, run_experiment
from geoai_open_lab.region_figures import save_figures
from IPython.display import SVG, display

places, aoi, config, provenance = load_inputs()
print({"release": config["release"], "run": config["input_run"],
       "rows": len(places), "sha256": config["input_sha256"]})

## 2. 吉祥寺駅周辺を1行の数値にする

内部で250m・500m格子を集計し、地域全体の件数と実面積から密度を算出します。地域の構成比はカテゴリ件数合計÷全登録件数です。区画ごとの構成比を単純平均しません。

比較候補は登録密度・カテゴリ構成比・250m集中度。面積・件数・欠損率とmanifestのrelease・取得日・AOI・格子条件は監査情報です。登録数と密度の二重評価を避けるため、実際の採用列・正規化・距離関数は次段階で決めます。

In [ ]:
import pandas as pd
results, manifest, output_dir = run_experiment()
profile = pd.read_csv(output_dir / "region_features.csv")
with pd.option_context("display.max_columns", None):
    display(profile)
# 横長の表を日本語・単位付きで読み直すプロフィール図
figures = save_figures(results, output_dir)
display(SVG(filename=str(figures[2])))

## 3. 各数値と、250m格子による算出根拠

- 規模：全登録数。集中：登録数÷AOI内面積(km²)。地域平均は格子密度の単純平均ではありません。
- 内訳：各群の構成比。表示群・その他・欠損を合わせて100%。定義はconfigs/region_features.tomlのdisplay_groupsが正本です。
- 集中度：250m格子の最大密度÷地域全体の平均密度。格子原点・端の小区画・AOIに影響されるため同条件で比較します。

以下の地図は算出根拠です。色は件/km²、数字は件数。500mも同じ入力で集計し、合計が一致することを確認します。

In [ ]:
display(SVG(filename=str(figures[0])))

250mで最多の区画は707件、密度11,312件/km²です。500mでは区画をまとめるため、集中の見え方が変わります。面積が小さい端の区画は率が変動しやすく、色だけで特徴を断定しません。

## 4. カテゴリ構成を見る

全25区画を北から南、同じ行は西から東へ並べます。表示用対応表は設定ファイルにあり、「その他」と「カテゴリ欠損」は別です。

In [ ]:
display(SVG(filename=str(figures[1])))

構成比は規模を表しません。同じ構成比でも件数が違うため、各区画のnと面積を併記しています。元のbasic_category別の表も保存済みです。

## 5. 欠損と営業状態を確認

欠損は0件でも営業中でもありません。

In [ ]:
print("カテゴリ欠損:", manifest["category_counts"].get("__MISSING__", 0))
print("営業状態:", manifest["status_counts"])
print("対象面積 (km²):", manifest["area_m2"] / 1_000_000)
print("250mと500mの合計:", {s: int(c.poi_count.sum()) for s, (c, _) in results.items()})

## まとめ

- 同一データでも、件数・密度・構成比は異なる側面を表します。
- カテゴリ欠損55件、営業状態欠損3,665件。営業中店舗数とは言えません。
- 今回0件区画はありませんが、実装は0件と未取得・カテゴリ欠損を区別します。
- 吉祥寺の実験用矩形内部だけの結果です。駅の縄張り・実際の商圏ではありません。
- 街同士の比較・普遍的地域分類・クラスタリング・Embeddingは後続実験です。

公開用Notebookには出力を残しません。実行済み版・HTML・図表はdata内へ保存してください。

主成果はregion_features.csvの1地域1行です。次回は同じ定義の列を複数地域に追加します。特徴量同士の重複や構成比の総和制約も考慮し、類似度に使う列はその時に決めます。